# PRUEBAS

In [0]:
"""
dbutils.widgets.removeAll()

dbutils.widgets.text("catalogName", "sbsrisk_dev")
dbutils.widgets.text("year", "2024")   # hoy trabajamos 2024, luego lo replicamos a 2025
dbutils.widgets.text("bronzeTable", "bronze_sbs_indice_2024")
dbutils.widgets.text("silverTable", "silver_sbs_indice_2024")

catalog = dbutils.widgets.get("catalogName")
year = dbutils.widgets.get("year")
bronze_table = f"{catalog}.bronze.{dbutils.widgets.get('bronzeTable')}"
silver_table = f"{catalog}.silver.{dbutils.widgets.get('silverTable')}"

print("bronze_table:", bronze_table)
print("silver_table:", silver_table)
"""

In [0]:
"""
from pyspark.sql.functions import col, trim, lower

df_bronze = spark.table(bronze_table)

display(df_bronze.limit(20))
print("rows bronze:", df_bronze.count())
"""

In [0]:
"""
from pyspark.sql.functions import col

df_clean = (
    df_bronze
    .filter(
        (col("_c0").isNotNull()) | (col("_c1").isNotNull()) | (col("_c2").isNotNull()) |
        (col("_c3").isNotNull()) | (col("_c4").isNotNull()) | (col("_c5").isNotNull()) | (col("_c6").isNotNull())
    )
)

display(df_clean.limit(30))
print("rows after non-null filter:", df_clean.count())
"""

In [0]:
"""
# buscamos filas donde _c0 tenga palabras tipo "Caja", "CMAC", "Código", etc.
from pyspark.sql.functions import lower

df_candidates = (
    df_clean
    .withColumn("_c0_l", lower(col("_c0")))
    .filter(
        col("_c0_l").contains("caja") |
        col("_c0_l").contains("cmac") |
        col("_c0_l").contains("codigo") |
        col("_c0_l").contains("código")
    )
    .select("_c0","_c1","_c2","_c3","_c4","_c5","_c6","source_file","periodo_mes")
)

display(df_candidates.limit(50))
"""

In [0]:
"""
spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA silver")

(
  df_clean.write.format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(silver_table)
)

print("OK -> saved:", silver_table)
"""

In [0]:
"""
%sql
SELECT COUNT(*) AS total_rows
FROM sbsrisk_dev.silver.silver_sbs_indice_2024;

DESCRIBE EXTENDED sbsrisk_dev.silver.silver_sbs_indice_2024;
"""

# INICIANDO


In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
USE CATALOG sbsrisk_dev;
USE SCHEMA silver;

In [0]:
from pyspark.sql import functions as F

df24 = spark.table("sbsrisk_dev.bronze.bronze_sbs_indice_2024").withColumn("anio", F.lit(2024))
df25 = spark.table("sbsrisk_dev.bronze.bronze_sbs_indice_2025").withColumn("anio", F.lit(2025))

df = df24.unionByName(df25, allowMissingColumns=True)

print("Total filas (bronze 2024+2025):", df.count())
display(df.limit(20))

In [0]:
from pyspark.sql import functions as F

df24 = spark.table("sbsrisk_dev.bronze.bronze_sbs_indice_2024").withColumn("anio", F.lit(2024))
df25 = spark.table("sbsrisk_dev.bronze.bronze_sbs_indice_2025").withColumn("anio", F.lit(2025))
df = df24.unionByName(df25, allowMissingColumns=True)

cols_excel = ["_c0","_c1","_c2","_c3","_c4","_c5","_c6"]

# conteo de celdas NO vacías por fila
df_chk = df.withColumn(
    "nn",
    sum([F.when(F.col(c).isNotNull() & (F.trim(F.col(c)) != ""), 1).otherwise(0) for c in cols_excel])
)

df_chk.groupBy("nn").count().orderBy("nn").show()
display(df_chk.orderBy(F.desc("nn")).select(*cols_excel, "source_file","periodo_mes","anio","nn").limit(50))

In [0]:
cols_excel = ["_c0","_c1","_c2","_c3","_c4","_c5","_c6"]

# 1) Quitar filas donde TODAS las columnas estén vacías
cond_all_null = None
for c in cols_excel:
    expr = F.col(c).isNull() | (F.trim(F.col(c)) == "")
    cond_all_null = expr if cond_all_null is None else (cond_all_null & expr)

df_clean = df.filter(~cond_all_null)

# 2) Quitar SOLO filas de “título”, pero sin botar las filas con _c0 NULL
titulo = F.lower(F.coalesce(F.col("_c0"), F.lit("")))
df_clean = df_clean.filter(~titulo.contains("boletin informativo mensual"))

# 3) Trim
for c in cols_excel + ["source_file", "periodo_mes", "anio"]:
    df_clean = df_clean.withColumn(c, F.trim(F.col(c)))

print("Total filas silver limpio:", df_clean.count())
display(df_clean.limit(30))

In [0]:
(
  df_clean.write.format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("sbsrisk_dev.silver.silver_sbs_indice")
)

In [0]:
%sql
SELECT anio, COUNT(*) AS filas
FROM sbsrisk_dev.silver.silver_sbs_indice
GROUP BY anio
ORDER BY anio;

SELECT periodo_mes, COUNT(*) AS filas
FROM sbsrisk_dev.silver.silver_sbs_indice
GROUP BY periodo_mes
ORDER BY periodo_mes;